# Build an STP Knowledge Graph — in Python

A tiny, **runnable** version of the *Model Spanning Tree as a Knowledge Graph* lesson.
It uses **Python + networkx** so it runs right here in Colab — no database to install.

**The one idea:** STP already computes a graph in every switch's head — a **root
bridge**, and a **forwarding/blocking** state on every port, so frames never loop.
A *knowledge graph* is that same machine, except **you** pick the nodes — so you can
add the one node STP never had: the **business service** riding on top of a VLAN.

**Is this machine learning?** No. It is just structured facts you can question — no
training, no guessing. If you can read `show spanning-tree`, you can read this.


## The words you need

- **root bridge** — the elected switch every other switch measures its path against.
- **BPDU** — the "hello" message switches trade to elect a root and spot topology changes.
- **port state: forwarding / blocking** — `forwarding` carries traffic; `blocking` is up
  but silent, a standby against loops.
- **STP instance / VLAN** — one tree per VLAN. This lesson folds the instance into the
  `VLAN` node itself, since in a small design it's almost always 1:1.
- **ROOT_FOR** — the edge naming which switch is root for a VLAN, right now.
- **REACHED_VIA** — the edge naming which access switch actually delivers a VLAN to endpoints.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# networkx and matplotlib come pre-installed in Colab.
# A DiGraph is a *directed* graph: every edge has a direction, like our arrows.
G = nx.DiGraph()
print('empty graph ready:', G)

## Step 1 — Create your first nodes

Three switches, one VLAN, one service. Each node gets a **label** (its type) and some
**properties** (facts about it) — `role`/`priority` on switches, `vlan_id` on the VLAN.


In [ ]:
# three switches: a root, a distribution switch, and an access switch
G.add_node('core-sw1', label='Switch', role='root',     priority=4096)
G.add_node('dist-sw1', label='Switch', role='non-root', priority=32768)
G.add_node('acc-sw1',  label='Switch', role='non-root', priority=32768)

# one VLAN (the STP instance is folded into this node, for simplicity)
G.add_node('VLAN 10', label='VLAN', vlan_id=10)

print(G.number_of_nodes(), 'nodes:', list(G.nodes))

## Step 2 — Connect them (the edge carries the port state)

`core-sw1` is elected **root** for VLAN 10. `acc-sw1` reaches the root two ways: a
**forwarding** uplink to `dist-sw1`, and a redundant, **blocking** uplink straight to
`core-sw1` — a hot standby STP is holding in reserve so there's no loop.


In [ ]:
G.add_edge('core-sw1', 'VLAN 10', rel='ROOT_FOR')

G.add_edge('acc-sw1',  'dist-sw1', rel='CONNECTS', state='forwarding')   # the active path
G.add_edge('acc-sw1',  'core-sw1', rel='CONNECTS', state='blocking')    # standby, no loop
G.add_edge('dist-sw1', 'core-sw1', rel='CONNECTS', state='forwarding')

G.add_edge('VLAN 10', 'acc-sw1', rel='REACHED_VIA')   # this is where endpoints plug in

for u, v, d in G.edges(data=True):
    extra = f"({d['state']})" if 'state' in d else ''
    print(f'{u} --{d["rel"]}{extra}--> {v}')

## Step 3 — Add the service

A **business service** rides on the VLAN. This is the link no `show spanning-tree`
output holds: a Layer 2 fact wired to a business fact.


In [ ]:
G.add_node('Payroll App', label='Service', criticality='critical')
G.add_edge('Payroll App', 'VLAN 10', rel='RUNS_ON')

print('graph now has', G.number_of_nodes(), 'nodes and', G.number_of_edges(), 'edges')

## See it

Colour the nodes by their label; draw the blocking (standby) link dashed and red —
same visual convention as a down link, because right now it isn't carrying traffic either.


In [ ]:
colors = {'Switch': '#3aa0ff', 'VLAN': '#0f7f74', 'Service': '#c8702a'}
node_colors = [colors.get(G.nodes[n].get('label'), '#cccccc') for n in G]
pos = nx.spring_layout(G, seed=3)   # seed keeps the layout stable

plt.figure(figsize=(9, 6))
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1800, alpha=0.92)
nx.draw_networkx_labels(G, pos, font_size=9)

blocking = [(u, v) for u, v, d in G.edges(data=True) if d.get('state') == 'blocking']
other    = [(u, v) for u, v, d in G.edges(data=True) if (u, v) not in blocking]
nx.draw_networkx_edges(G, pos, edgelist=other,    arrows=True, arrowsize=16, edge_color='#8894a0')
nx.draw_networkx_edges(G, pos, edgelist=blocking, arrows=True, arrowsize=16, edge_color='#d34b3f', style='dashed')
nx.draw_networkx_edge_labels(G, pos, font_size=7,
    edge_labels={(u, v): d['rel'] for u, v, d in G.edges(data=True)})

plt.axis('off'); plt.title('STP knowledge graph'); plt.tight_layout(); plt.show()

## Step 4 — Ask it the 3 AM question

*If an access switch loses its only forwarding path, which services on that VLAN are at risk?*

We **traverse**: find an access switch reached by a VLAN that has **no** `forwarding`
`CONNECTS` edge left -> walk back to that VLAN -> the services `RUNS_ON` it.


In [ ]:
def blast_radius(G):
    """Services put at risk by any access switch with zero forwarding paths left."""
    at_risk = []
    for vlan, acc, d in G.edges(data=True):
        # 1) a VLAN reached via some access switch
        if d.get('rel') != 'REACHED_VIA':
            continue
        # 2) does that access switch still have ANY forwarding CONNECTS edge?
        has_forwarding = any(
            d2.get('rel') == 'CONNECTS' and d2.get('state') == 'forwarding'
            for _, _, d2 in G.out_edges(acc, data=True)
        )
        if has_forwarding:
            continue
        # 3) no forwarding path left -> which services ride this VLAN?
        for svc, _, d3 in G.in_edges(vlan, data=True):
            if d3.get('rel') == 'RUNS_ON':
                at_risk.append((acc, vlan, svc))
    return at_risk

result = blast_radius(G)
print('at risk:', result or 'nothing — acc-sw1 still has a forwarding path')

The switch only ever told you a port is blocking — that's normal, expected, by design.
Your graph just told you **nothing** is at risk yet, because `acc-sw1` still forwards
through `dist-sw1`. Watch what happens when that last good path fails.


## What-if: break the last forwarding path, then restore it

Flip `acc-sw1`'s link to `dist-sw1` to `blocking` — its only forwarding path is gone,
and (because this models the risky window before STP reconverges) the standby link to
`core-sw1` is still sitting in `blocking` too. Ask again: the **Payroll App** appears.
Set it back to `forwarding` — it disappears. You are running "what if this fails?" on
a model, safely.


In [ ]:
G['acc-sw1']['dist-sw1']['state'] = 'blocking'   # the last forwarding path just failed
print('after failure:', [svc for *_, svc in blast_radius(G)] or 'nothing at risk')

G['acc-sw1']['dist-sw1']['state'] = 'forwarding'  # path restored
print('after restore:', [svc for *_, svc in blast_radius(G)] or 'nothing at risk')

## Make it yours

Change the four values below to **one** VLAN, **one** access switch, and **one** service
*you* run, then re-run. Your own service name comes back from `blast_radius(G)` — that is
the moment the tool is yours. (Model the smallest slice that answers one real question;
you can always add one more node later.)


In [ ]:
# --- change these four values to your network ---
MY_VLAN    = 'VLAN 42'
MY_ACCESS  = 'campus-acc-1'
MY_ID      = 42
MY_SERVICE = 'Badge Reader Service'
# --------------------------------------------------

G.add_node(MY_VLAN,   label='VLAN',    vlan_id=MY_ID)
G.add_node(MY_ACCESS, label='Switch',  role='non-root', priority=32768)
G.add_node(MY_SERVICE, label='Service', criticality='critical')

G.add_edge(MY_ACCESS, 'core-sw1', rel='CONNECTS', state='blocking')  # the failed path you're modelling
G.add_edge(MY_VLAN,   MY_ACCESS,  rel='REACHED_VIA')
G.add_edge(MY_SERVICE, MY_VLAN,   rel='RUNS_ON')

for acc, vlan, svc in blast_radius(G):
    print(f'{acc} has no forwarding path  ->  AT RISK: {svc}')

## You built a knowledge graph

From an empty graph to a question no `show spanning-tree` output can answer. The
shapes you used — *a service runs on a VLAN, a VLAN is reached via an access switch,
a switch connects to another switch with a port state* — are the same ones every
bigger graph is made of.

**Where next:** the full lesson explains the *why* in depth, and the companion lab does
this exact thing in **Neo4j + Cypher** (a real graph database) instead of networkx —
same nodes, same edges, same blast-radius question.
